# Heuristic Component-Hybrid SLR Projections

Produces a defensible SLR projection figure that brackets the likely range:
- **IPCC** (lower bound): known to underestimate ice sheet contributions
- **Quadratic altimetry extrapolation** (reference): naive extension of recent trends
- **Component hybrid** (best estimate): calibrated thermosteric + glaciers, Aschwanden Greenland, A4 WAIS framework

The aggregate DOLS polynomial is too high because a single quadratic absorbs the shifting
component mix (emergence of land ice as dominant contributor) as spurious acceleration.
The component-wise approach constrains each contributor with its own physics.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
from pathlib import Path
from scipy import interpolate, stats
import sys
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

# Local imports
sys.path.insert(0, str(Path('.').resolve()))
from slr_data_readers import read_ipcc_ar6_component, read_imbie_antarctic_peninsula

# ══════════════════════════════════════════════════════════════════
#  USER CONFIGURATION
# ══════════════════════════════════════════════════════════════════
SSPS_TO_PLOT = ['SSP2-4.5']                # time-series SSPs
SSPS_HISTOGRAM = ['SSP2-4.5']              # histogram SSPs
HISTOGRAM_YEARS = [2060, 2100]             # years for histograms
N_SAMPLES = 5000                           # MC ensemble size
BASELINE_YEAR = 2005.0
GLACIER_CAP_MM = None                      # set to e.g. 250 to cap
WAIS_CALVING_FACTOR = 1.25                 # 25% uplift for missing calving
SEED = 42

# SSP name mappings
SSP_HDF5_TEMP = {
    'SSP1-1.9': 'SSP1_1_9', 'SSP1-2.6': 'SSP1_2_6',
    'SSP2-4.5': 'SSP2_4_5', 'SSP3-7.0': 'SSP3_7_0',
    'SSP5-8.5': 'SSP5_8_5',
}
SSP_HDF5_GMSL = {
    'SSP1-1.9': 'ssp119', 'SSP1-2.6': 'ssp126',
    'SSP2-4.5': 'ssp245', 'SSP3-7.0': 'ssp370',
    'SSP5-8.5': 'ssp585',
}

# Aschwanden et al. 2019 Greenland projections (mm at 2100, median [p5, p95])
ASCHWANDEN_GREENLAND = {
    'SSP1-2.6': {'median': 80, 'p5': 40, 'p95': 140},
    'SSP2-4.5': {'median': 110, 'p5': 50, 'p95': 180},
    'SSP3-7.0': {'median': 150, 'p5': 70, 'p95': 240},
    'SSP5-8.5': {'median': 190, 'p5': 90, 'p95': 290},
}

# Quadratic altimetry fit (from predictability_analysis.ipynb)
QUAD_ACCEL = 0.079   # mm/yr^2
QUAD_RATE_2005 = 3.25  # mm/yr at 2005
QUAD_2100_MEDIAN = 670  # mm
QUAD_2100_P5 = 345
QUAD_2100_P95 = 996

rng = np.random.default_rng(SEED)
print('Configuration loaded.')

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  DATA LOADING
# ══════════════════════════════════════════════════════════════════
data_dir = Path('..') / 'data'
h5_path = data_dir / 'processed' / 'slr_processed_data.h5'

# Stage 1 calibration coefficients (units: mm/yr/°C^n)
with open(data_dir / 'processed' / 'stage1_thermosteric_results.json') as f:
    stage1 = json.load(f)

# A4 WAIS quantiles
with open(data_dir / 'processed' / 'results_summary.json') as f:
    results_summary = json.load(f)

# ── Frederikse GMSL observations ──
df_fred = pd.read_hdf(h5_path, key='raw/df_frederikse')
fred_years = df_fred['year'].values.astype(float)
fred_gmsl_m = df_fred['gmsl'].values  # meters
fred_sigma_m = df_fred['gmsl_sigma'].values  # meters

# Rebase to 2005
idx_2005 = np.argmin(np.abs(fred_years - BASELINE_YEAR))
fred_gmsl_mm = (fred_gmsl_m - fred_gmsl_m[idx_2005]) * 1000  # mm
fred_sigma_mm = fred_sigma_m * 1000

print(f'Frederikse GMSL: {fred_years[0]:.0f}-{fred_years[-1]:.0f}, '
      f'{len(fred_years)} years')

# ── Temperature projections ──
temp_hist = pd.read_hdf(h5_path, key='projections/temp/Historical')
temp_ssps = {}
for ssp_name, ssp_key in SSP_HDF5_TEMP.items():
    try:
        temp_ssps[ssp_name] = pd.read_hdf(h5_path,
                                           key=f'projections/temp/{ssp_key}')
    except KeyError:
        pass
print(f'Temperature: Historical {temp_hist["decimal_year"].iloc[0]:.0f}-'
      f'{temp_hist["decimal_year"].iloc[-1]:.0f}, '
      f'{len(temp_ssps)} SSPs loaded')

# ── IPCC GMSL projections ──
ipcc_gmsl = {}
for ssp_name, ssp_code in SSP_HDF5_GMSL.items():
    try:
        df = pd.read_hdf(h5_path, key=f'projections/gmsl/{ssp_code}')
        ipcc_gmsl[ssp_name] = df
    except KeyError:
        pass
print(f'IPCC GMSL: {len(ipcc_gmsl)} SSPs loaded')

# ── IMBIE Antarctic Peninsula (use Gt file for correct column names) ──
pen_gt_path = data_dir / 'raw' / 'ice_sheets' / 'Gt' / \
              'imbie_antarctic_peninsula_2021_Gt.csv'
if pen_gt_path.exists():
    df_pen = read_imbie_antarctic_peninsula(str(pen_gt_path),
                                            convert_to_sle=True)
    print(f'Peninsula IMBIE: {len(df_pen)} points, '
          f'columns: {list(df_pen.columns)}')
else:
    df_pen = None
    print('Peninsula IMBIE: not found, will use default estimate')

# ── IPCC GIS component (for Greenland shape) ──
ipcc_component_dir = str(data_dir / 'raw' / 'ipcc_ar6' / 'slr' / 'ar6' /
                         'global' / 'dist_components')
ipcc_gis = {}
for ssp_name in SSPS_TO_PLOT + SSPS_HISTOGRAM:
    ssp_code = SSP_HDF5_GMSL.get(ssp_name, 'ssp245')
    try:
        df_gis = read_ipcc_ar6_component(
            ipcc_component_dir, 'icesheets', sub_component='GIS',
            scenario=ssp_code, convert_to_meters=False)  # mm
        ipcc_gis[ssp_name] = df_gis
    except Exception as e:
        print(f'  Warning: Could not load IPCC GIS for {ssp_name}: {e}')

print(f'IPCC GIS component: {len(ipcc_gis)} SSPs loaded')
print('\nAll data loaded successfully.')

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  BUILD TEMPERATURE TIMELINES + DESIGN VECTORS
# ══════════════════════════════════════════════════════════════════

def build_projection_temperature(temp_hist, temp_ssp, baseline_year=2005.0):
    """Merge historical + SSP temperature into one annual series from baseline.
    Returns (years, T_anomaly) starting at baseline_year."""
    # Historical
    t_h = temp_hist['decimal_year'].values
    T_h = temp_hist['temperature'].values
    # SSP
    t_s = temp_ssp['decimal_year'].values
    T_s = temp_ssp['temperature'].values

    # Splice: use historical up to 2014, SSP from 2015
    mask_h = t_h < 2015
    mask_s = t_s >= 2015
    years = np.concatenate([t_h[mask_h], t_s[mask_s]])
    T = np.concatenate([T_h[mask_h], T_s[mask_s]])

    # Trim to start at baseline
    mask_bl = years >= baseline_year
    return years[mask_bl], T[mask_bl]


def compute_design_vectors(years, T):
    """Compute cumulative design vectors from annual temperature.
    Returns DeltaI2, DeltaI1, DeltaI0 relative to first year (= baseline)."""
    n = len(T)
    dt = np.diff(years)
    I2 = np.zeros(n)
    I1 = np.zeros(n)
    for i in range(n - 1):
        I2[i + 1] = I2[i] + 0.5 * (T[i]**2 + T[i + 1]**2) * dt[i]
        I1[i + 1] = I1[i] + 0.5 * (T[i] + T[i + 1]) * dt[i]
    I0 = years - years[0]
    return I2, I1, I0


# Build for each SSP
all_ssps = set(SSPS_TO_PLOT) | set(SSPS_HISTOGRAM)
proj_temp = {}  # {ssp: (years, T, I2, I1, I0)}

for ssp in all_ssps:
    if ssp not in temp_ssps:
        print(f'  Skipping {ssp}: no temperature data')
        continue
    yrs, T = build_projection_temperature(temp_hist, temp_ssps[ssp],
                                          BASELINE_YEAR)
    I2, I1, I0 = compute_design_vectors(yrs, T)
    proj_temp[ssp] = {'years': yrs, 'T': T, 'I2': I2, 'I1': I1, 'I0': I0}
    print(f'  {ssp}: {yrs[0]:.0f}-{yrs[-1]:.0f}, '
          f'T range [{T[0]:.2f}, {T[-1]:.2f}] K')

print('Design vectors computed.')

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  COMPONENT MC PROJECTIONS
# ══════════════════════════════════════════════════════════════════

def sample_coefficients(coeff_dict, N, rng):
    """Sample from approximate Gaussian using median and 90% CI.
    Returns (N,) array of samples."""
    mu = coeff_dict['median']
    lo, hi = coeff_dict['ci_90']
    sigma = (hi - lo) / (2 * 1.645)
    return rng.normal(mu, max(sigma, 1e-10), size=N)


def project_polynomial(I2, I1, I0, a_samples, b_samples, c_samples):
    """Compute H(t) = a*I2 + b*I1 + c*I0 for each MC sample.
    Returns (N, n_time) array in same units as coefficients."""
    # (N, 1) x (1, n_time) -> (N, n_time)
    H = (a_samples[:, None] * I2[None, :]
         + b_samples[:, None] * I1[None, :]
         + c_samples[:, None] * I0[None, :])
    return H


# ── 1. THERMOSTERIC ──
print('1. Thermosteric (calibrated Bayesian fit)...')
cal_th = stage1['thermosteric_calibration']
a_th = sample_coefficients(cal_th['a_th'], N_SAMPLES, rng)
b_th = sample_coefficients(cal_th['b_th'], N_SAMPLES, rng)
c_th = sample_coefficients(cal_th['c_th'], N_SAMPLES, rng)
# a_th is for quadratic (T^2) term; enforce non-negative
a_th = np.maximum(a_th, 0)

thermo_samples = {}  # {ssp: (N, n_time) in mm}
for ssp in all_ssps:
    pt = proj_temp[ssp]
    H = project_polynomial(pt['I2'], pt['I1'], pt['I0'],
                           a_th, b_th, c_th)
    thermo_samples[ssp] = H
    med_2100 = np.median(H[:, -1])
    print(f'  {ssp} at 2100: {med_2100:.0f} mm '
          f'[{np.percentile(H[:, -1], 5):.0f}, '
          f'{np.percentile(H[:, -1], 95):.0f}]')


# ── 2. GLACIERS ──
print('\n2. Glaciers (calibrated + optional cap)...')
cal_gl = stage1['glacier_calibration']
a_gl = sample_coefficients(cal_gl['a_gl'], N_SAMPLES, rng)
b_gl = sample_coefficients(cal_gl['b_gl'], N_SAMPLES, rng)
c_gl = sample_coefficients(cal_gl['c_gl'], N_SAMPLES, rng)
a_gl = np.maximum(a_gl, 0)

glacier_samples = {}
for ssp in all_ssps:
    pt = proj_temp[ssp]
    H = project_polynomial(pt['I2'], pt['I1'], pt['I0'],
                           a_gl, b_gl, c_gl)
    if GLACIER_CAP_MM is not None:
        H = np.minimum(H, GLACIER_CAP_MM)
    glacier_samples[ssp] = H
    med_2100 = np.median(H[:, -1])
    cap_str = f' (capped at {GLACIER_CAP_MM})' if GLACIER_CAP_MM else ''
    print(f'  {ssp} at 2100: {med_2100:.0f} mm '
          f'[{np.percentile(H[:, -1], 5):.0f}, '
          f'{np.percentile(H[:, -1], 95):.0f}]{cap_str}')


# ── 3. GREENLAND (Aschwanden + IPCC GIS shape) ──
print('\n3. Greenland (Aschwanden et al. 2019)...')

greenland_samples = {}
for ssp in all_ssps:
    pt = proj_temp[ssp]
    n_time = len(pt['years'])

    # Aschwanden endpoint at 2100
    asch = ASCHWANDEN_GREENLAND.get(ssp, ASCHWANDEN_GREENLAND['SSP2-4.5'])
    # Fit log-normal: median = exp(mu), solve for sigma from p5/p95
    mu_ln = np.log(max(asch['median'], 1))
    # sigma from p5 to p95 range on log scale
    sigma_ln = (np.log(max(asch['p95'], 1)) -
                np.log(max(asch['p5'], 1))) / (2 * 1.645)
    sigma_ln = max(sigma_ln, 0.1)
    gris_2100 = rng.lognormal(mu_ln, sigma_ln, size=N_SAMPLES)

    # Temporal shape from IPCC GIS (decadal, interpolate to annual)
    if ssp in ipcc_gis and len(ipcc_gis[ssp]) > 0:
        gis_years = ipcc_gis[ssp].index.values.astype(float)
        gis_median = ipcc_gis[ssp]['median'].values * 1000  # m -> mm
        # Normalize shape to [0, 1] at 2100
        idx_2100 = np.argmin(np.abs(gis_years - 2100))
        gis_norm = gis_median / max(gis_median[idx_2100], 1e-6)
        gis_norm = np.maximum(gis_norm, 0)
        # Interpolate to projection years
        shape_fn = interpolate.interp1d(
            gis_years, gis_norm, kind='cubic',
            bounds_error=False, fill_value=(0, gis_norm[-1]))
        shape = shape_fn(pt['years'])
        shape = np.maximum(shape, 0)
    else:
        # Fallback: quadratic growth shape
        dt = pt['years'] - BASELINE_YEAR
        shape = (dt / (2100 - BASELINE_YEAR))**2

    # MC: each sample gets its 2100 value, distributed by shape
    H = gris_2100[:, None] * shape[None, :]
    greenland_samples[ssp] = H
    med_2100 = np.median(H[:, -1])
    print(f'  {ssp} at 2100: {med_2100:.0f} mm '
          f'[{np.percentile(H[:, -1], 5):.0f}, '
          f'{np.percentile(H[:, -1], 95):.0f}]')


# ── 4. WEST ANTARCTICA (A4 framework + calving factor) ──
print('\n4. West Antarctica (A4 + 25% calving)...')

a4 = results_summary['a4_ice_uncertainty']
# Time points and their quantiles
a4_years = np.array([2005, 2020, 2030, 2050, 2070, 2100])
a4_medians = np.array([0, 0,
                        a4['2030']['median_mm'],
                        a4['2050']['median_mm'],
                        a4['2070']['median_mm'],
                        a4['2100']['median_mm']])

# Fit log-normal at 2100 from quantiles
med_100 = a4['2100']['median_mm']
p5_100 = max(a4['2100']['p05_mm'], 1)
p95_100 = a4['2100']['p95_mm']
mu_wais = np.log(max(med_100, 1))
sigma_wais = (np.log(p95_100) - np.log(p5_100)) / (2 * 1.645)
sigma_wais = max(sigma_wais, 0.1)

# Sample WAIS at 2100
wais_2100 = rng.lognormal(mu_wais, sigma_wais, size=N_SAMPLES)
wais_2100 *= WAIS_CALVING_FACTOR

# Temporal shape from A4 median ratios
a4_shape = a4_medians / max(a4_medians[-1], 1e-6)  # [0, 0, 0, 0.26, 0.56, 1]
shape_fn_wais = interpolate.interp1d(
    a4_years, a4_shape, kind='linear',
    bounds_error=False, fill_value=(0, 1))

wais_samples = {}
for ssp in all_ssps:
    pt = proj_temp[ssp]
    shape = shape_fn_wais(pt['years'])
    shape = np.maximum(shape, 0)
    H = wais_2100[:, None] * shape[None, :]
    wais_samples[ssp] = H
    med = np.median(H[:, -1])
    print(f'  {ssp} at 2100: {med:.0f} mm '
          f'[{np.percentile(H[:, -1], 5):.0f}, '
          f'{np.percentile(H[:, -1], 95):.0f}]')


# ── 5. ANTARCTIC PENINSULA (IMBIE temperature sensitivity) ──
print('\n5. Antarctic Peninsula (IMBIE rate extrapolation)...')

if df_pen is not None:
    # Compute linear rate from IMBIE cumulative (already in m SLE)
    pen_dec_yr = df_pen['decimal_year'].values
    pen_cum = df_pen['cumulative_mass_balance'].values  # m SLE
    pen_cum_mm = pen_cum * 1000  # mm
    # Annual averages
    pen_ann_yr = np.arange(1993, 2021)
    pen_ann_cum = np.array([
        np.mean(pen_cum_mm[(pen_dec_yr >= yr) & (pen_dec_yr < yr + 1)])
        for yr in pen_ann_yr
    ])
    # Linear fit: cumulative = rate * (t - 2005) + offset
    mask_valid = ~np.isnan(pen_ann_cum)
    pen_fit = np.polyfit(pen_ann_yr[mask_valid] - BASELINE_YEAR,
                         pen_ann_cum[mask_valid], 1)
    pen_rate = pen_fit[0]  # mm/yr (slope of cumulative = rate)
    pen_rate_sigma = abs(pen_rate) * 0.3  # 30% uncertainty
    print(f'  Peninsula rate: {pen_rate:.2f} +/- {pen_rate_sigma:.2f} mm/yr')
else:
    pen_rate = 0.5  # ~0.5 mm/yr default estimate
    pen_rate_sigma = 0.3
    print(f'  Peninsula rate (default): {pen_rate:.2f} mm/yr')

# Project peninsula: linear, with uncertainty on rate
pen_rates_mc = rng.normal(pen_rate, pen_rate_sigma, size=N_SAMPLES)
peninsula_samples = {}
for ssp in all_ssps:
    pt = proj_temp[ssp]
    dt = pt['years'] - BASELINE_YEAR
    # Cumulative = rate * dt + 0.5 * rate_change * dt^2
    # For simplicity, linear: H = rate * dt
    H = pen_rates_mc[:, None] * dt[None, :]
    peninsula_samples[ssp] = H

med_pen_2100 = np.median(peninsula_samples[list(all_ssps)[0]][:, -1])
print(f'  At 2100: ~{med_pen_2100:.0f} mm')


# ── 6. EAST ANTARCTICA (assume balance) ──
print('\n6. East Antarctica: ~0 mm (assumed in balance)')
eais_samples = {}
for ssp in all_ssps:
    pt = proj_temp[ssp]
    H = rng.normal(0, 10, size=(N_SAMPLES, len(pt['years'])))
    # Make it cumulative-like (random walk scaled to +/-20mm at 2100)
    H = np.cumsum(H, axis=1) * 0.2  # small random contribution
    eais_samples[ssp] = H


# ── 7. TERRESTRIAL WATER STORAGE (IPCC) ──
print('\n7. TWS (from IPCC GMSL component)...')
tws_samples = {}
for ssp in all_ssps:
    pt = proj_temp[ssp]
    ssp_code = SSP_HDF5_GMSL.get(ssp, 'ssp245')
    if ssp in ipcc_gmsl:
        ipcc_df = ipcc_gmsl[ssp]
        ipcc_yrs = ipcc_df['decimal_year'].values
        ipcc_tws_m = ipcc_df['landwaterstorage'].values  # meters
        ipcc_tws_mm = ipcc_tws_m * 1000
        # Interpolate to projection years
        tws_fn = interpolate.interp1d(
            ipcc_yrs, ipcc_tws_mm, kind='linear',
            bounds_error=False, fill_value=(0, ipcc_tws_mm[-1]))
        tws_median = tws_fn(pt['years'])
        tws_median = np.maximum(tws_median, 0)
        # Add uncertainty (~30% relative)
        tws_scale = rng.normal(1.0, 0.3, size=N_SAMPLES)
        tws_scale = np.maximum(tws_scale, 0)
        H = tws_scale[:, None] * tws_median[None, :]
    else:
        # Fallback: linear 50 mm by 2100
        dt = pt['years'] - BASELINE_YEAR
        rate = 50 / (2100 - BASELINE_YEAR)
        tws_scale = rng.normal(1.0, 0.3, size=N_SAMPLES)
        H = tws_scale[:, None] * rate * dt[None, :]
    tws_samples[ssp] = H
    med_tws = np.median(H[:, -1])
    print(f'  {ssp} at 2100: {med_tws:.0f} mm')

print('\nAll components projected.')

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  COMPOSITE ASSEMBLY
# ══════════════════════════════════════════════════════════════════

hybrid_results = {}  # {ssp: dict with years, median, bands, samples}

for ssp in all_ssps:
    pt = proj_temp[ssp]
    years = pt['years']

    # Sum all components (N_SAMPLES, n_time)
    total = (thermo_samples[ssp]
             + glacier_samples[ssp]
             + greenland_samples[ssp]
             + wais_samples[ssp]
             + peninsula_samples[ssp]
             + eais_samples[ssp]
             + tws_samples[ssp])

    hybrid_results[ssp] = {
        'years': years,
        'samples': total,
        'median': np.median(total, axis=0),
        'p5': np.percentile(total, 5, axis=0),
        'p17': np.percentile(total, 17, axis=0),
        'p83': np.percentile(total, 83, axis=0),
        'p95': np.percentile(total, 95, axis=0),
        # Component medians for stacking
        'thermo_med': np.median(thermo_samples[ssp], axis=0),
        'glacier_med': np.median(glacier_samples[ssp], axis=0),
        'greenland_med': np.median(greenland_samples[ssp], axis=0),
        'wais_med': np.median(wais_samples[ssp], axis=0),
        'peninsula_med': np.median(peninsula_samples[ssp], axis=0),
        'tws_med': np.median(tws_samples[ssp], axis=0),
    }

    print(f'{ssp} at 2100: median={hybrid_results[ssp]["median"][-1]:.0f} mm '
          f'[{hybrid_results[ssp]["p5"][-1]:.0f}, '
          f'{hybrid_results[ssp]["p95"][-1]:.0f}]')

# ── IPCC total (interpolated to annual) ──
ipcc_annual = {}
for ssp in all_ssps:
    if ssp not in ipcc_gmsl:
        continue
    df = ipcc_gmsl[ssp]
    yrs_ipcc = df['decimal_year'].values
    keys = ['gmsl', 'gmsl_lower', 'gmsl_upper', 'gmsl_17', 'gmsl_83']
    interp = {}
    for k in keys:
        fn = interpolate.interp1d(
            yrs_ipcc, df[k].values * 1000,  # m -> mm
            kind='cubic', bounds_error=False,
            fill_value=(0, df[k].values[-1] * 1000))
        interp[k] = fn
    # Evaluate on hybrid years
    pt = proj_temp[ssp]
    ipcc_annual[ssp] = {
        'years': pt['years'],
        'median': interp['gmsl'](pt['years']),
        'p5': interp['gmsl_lower'](pt['years']),
        'p95': interp['gmsl_upper'](pt['years']),
        'p17': interp['gmsl_17'](pt['years']),
        'p83': interp['gmsl_83'](pt['years']),
    }

# ── Quadratic altimetry extrapolation ──
def quad_extrapolation(years, baseline=BASELINE_YEAR):
    """Quadratic extrapolation from satellite altimetry fit."""
    dt = years - baseline
    median = 0.5 * QUAD_ACCEL * dt**2 + QUAD_RATE_2005 * dt
    # Uncertainty grows as dt^2
    sigma_2100 = (QUAD_2100_P95 - QUAD_2100_P5) / (2 * 1.645)
    dt_2100 = 2100 - baseline
    sigma = sigma_2100 * (dt / dt_2100)**2
    sigma = np.maximum(sigma, 1)  # floor at 1 mm
    return median, sigma

print('\nComposite assembly complete.')

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  FIGURE 1: TIME SERIES (2000-2100)
# ══════════════════════════════════════════════════════════════════

fig, ax = plt.subplots(1, 1, figsize=(12, 7))

ssp_colors = {
    'SSP1-2.6': '#2166ac', 'SSP2-4.5': '#d6604d',
    'SSP3-7.0': '#b2182b', 'SSP5-8.5': '#67001f',
}

for ssp in SSPS_TO_PLOT:
    if ssp not in hybrid_results:
        continue
    hr = hybrid_results[ssp]
    ia = ipcc_annual.get(ssp)
    color = ssp_colors.get(ssp, '#d6604d')
    years = hr['years']

    # ── IPCC band ──
    if ia is not None:
        ax.fill_between(ia['years'], ia['p5'], ia['p95'],
                        alpha=0.15, color='gray', zorder=1)
        ax.fill_between(ia['years'], ia['p17'], ia['p83'],
                        alpha=0.25, color='gray', zorder=1)
        ax.plot(ia['years'], ia['median'], color='gray', lw=1.5,
                ls='--', zorder=2, label=f'IPCC AR6 {ssp}')

    # ── Quadratic extrapolation ──
    q_med, q_sig = quad_extrapolation(years)
    ax.fill_between(years, q_med - 1.645 * q_sig,
                    q_med + 1.645 * q_sig,
                    alpha=0.1, color='orange', zorder=1)
    ax.plot(years, q_med, color='orange', lw=1.5, ls=':',
            zorder=2, label='Quadratic trend extrapolation')

    # ── Component hybrid band ──
    ax.fill_between(years, hr['p5'], hr['p95'],
                    alpha=0.2, color=color, zorder=3)
    ax.fill_between(years, hr['p17'], hr['p83'],
                    alpha=0.35, color=color, zorder=3)
    ax.plot(years, hr['median'], color=color, lw=2.5,
            zorder=4, label=f'Component hybrid {ssp}')

# ── Observations ──
obs_mask = fred_years >= 2000
ax.fill_between(fred_years[obs_mask],
                fred_gmsl_mm[obs_mask] - 1.645 * fred_sigma_mm[obs_mask],
                fred_gmsl_mm[obs_mask] + 1.645 * fred_sigma_mm[obs_mask],
                alpha=0.3, color='black', zorder=5)
ax.plot(fred_years[obs_mask], fred_gmsl_mm[obs_mask],
        color='black', lw=2, zorder=6, label='Observed (Frederikse)')

ax.set_xlabel('Year', fontsize=13)
ax.set_ylabel('Global Mean Sea Level Rise (mm, relative to 2005)', fontsize=13)
ax.set_xlim(2000, 2100)
ax.set_ylim(-50, None)
ax.legend(loc='upper left', fontsize=10, framealpha=0.9)
ax.grid(True, alpha=0.3)
ax.set_title('Global Mean Sea-Level Rise: Component-Hybrid Projection',
             fontsize=14, fontweight='bold')

plt.tight_layout()
fig.savefig('../figures/heuristic_slr_projection_timeseries.png',
            dpi=200, bbox_inches='tight')
print('Figure saved: heuristic_slr_projection_timeseries.png')
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  FIGURE 2: HISTOGRAMS AT 2060 AND 2100
# ══════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, len(HISTOGRAM_YEARS), figsize=(14, 5),
                          sharey=True)
if len(HISTOGRAM_YEARS) == 1:
    axes = [axes]

for ax, yr in zip(axes, HISTOGRAM_YEARS):
    for ssp in SSPS_HISTOGRAM:
        if ssp not in hybrid_results:
            continue
        hr = hybrid_results[ssp]
        color = ssp_colors.get(ssp, '#d6604d')

        # Find closest time index
        idx = np.argmin(np.abs(hr['years'] - yr))

        # ── IPCC distribution (approximate as Gaussian) ──
        if ssp in ipcc_annual:
            ia = ipcc_annual[ssp]
            idx_ipcc = np.argmin(np.abs(ia['years'] - yr))
            ipcc_med = ia['median'][idx_ipcc]
            ipcc_sigma = (ia['p95'][idx_ipcc] - ia['p5'][idx_ipcc]) / \
                         (2 * 1.645)
            x_ipcc = np.linspace(ipcc_med - 3 * ipcc_sigma,
                                 ipcc_med + 3 * ipcc_sigma, 300)
            pdf_ipcc = stats.norm.pdf(x_ipcc, ipcc_med, ipcc_sigma)
            ax.fill_between(x_ipcc, pdf_ipcc, alpha=0.25, color='gray')
            ax.plot(x_ipcc, pdf_ipcc, color='gray', lw=1.5, ls='--',
                    label='IPCC AR6' if yr == HISTOGRAM_YEARS[0] else '')
            ax.axvline(ipcc_med, color='gray', ls='--', alpha=0.7, lw=1)

        # ── Quadratic extrapolation distribution ──
        q_med_yr, q_sig_yr = quad_extrapolation(np.array([yr]))
        q_med_val = q_med_yr[0]
        q_sig_val = q_sig_yr[0]
        x_quad = np.linspace(q_med_val - 3 * q_sig_val,
                             q_med_val + 3 * q_sig_val, 300)
        pdf_quad = stats.norm.pdf(x_quad, q_med_val, q_sig_val)
        ax.plot(x_quad, pdf_quad, color='orange', lw=1.5, ls=':',
                label='Quadratic trend' if yr == HISTOGRAM_YEARS[0] else '')
        ax.axvline(q_med_val, color='orange', ls=':', alpha=0.7, lw=1)

        # ── Component hybrid KDE ──
        hybrid_vals = hr['samples'][:, idx]
        # Clip extreme outliers for cleaner KDE
        p1, p99 = np.percentile(hybrid_vals, [1, 99])
        hybrid_clip = hybrid_vals[(hybrid_vals >= p1) & (hybrid_vals <= p99)]
        kde = stats.gaussian_kde(hybrid_clip)
        x_hybrid = np.linspace(p1, p99, 300)
        pdf_hybrid = kde(x_hybrid)
        ax.fill_between(x_hybrid, pdf_hybrid, alpha=0.35, color=color)
        ax.plot(x_hybrid, pdf_hybrid, color=color, lw=2,
                label='Component hybrid' if yr == HISTOGRAM_YEARS[0] else '')
        hybrid_med = np.median(hybrid_vals)
        ax.axvline(hybrid_med, color=color, ls='-', alpha=0.7, lw=1.5)

    ax.set_xlabel('GMSL Rise (mm, relative to 2005)', fontsize=12)
    ax.set_title(f'{yr}', fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.2)

axes[0].set_ylabel('Probability Density', fontsize=12)
axes[0].legend(loc='upper right', fontsize=9, framealpha=0.9)

ssp_str = ', '.join(SSPS_HISTOGRAM)
fig.suptitle(f'GMSL Distribution at Key Horizons ({ssp_str})',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
fig.savefig('../figures/heuristic_slr_projection_histograms.png',
            dpi=200, bbox_inches='tight')
print('Figure saved: heuristic_slr_projection_histograms.png')
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  SUMMARY TABLE
# ══════════════════════════════════════════════════════════════════

print('=' * 80)
print('GMSL Projections Summary (mm, relative to 2005)')
print('=' * 80)

for ssp in sorted(all_ssps):
    if ssp not in hybrid_results:
        continue
    hr = hybrid_results[ssp]

    for yr in [2050, 2060, 2080, 2100]:
        idx = np.argmin(np.abs(hr['years'] - yr))

        # Hybrid
        h_med = hr['median'][idx]
        h_p5 = hr['p5'][idx]
        h_p95 = hr['p95'][idx]

        # IPCC
        if ssp in ipcc_annual:
            ia = ipcc_annual[ssp]
            idx_i = np.argmin(np.abs(ia['years'] - yr))
            i_med = ia['median'][idx_i]
            i_p5 = ia['p5'][idx_i]
            i_p95 = ia['p95'][idx_i]
        else:
            i_med = i_p5 = i_p95 = np.nan

        # Quadratic
        q_med, q_sig = quad_extrapolation(np.array([yr]))
        q_lo = q_med[0] - 1.645 * q_sig[0]
        q_hi = q_med[0] + 1.645 * q_sig[0]

        print(f'\n{ssp} at {yr}:')
        print(f'  IPCC AR6:      {i_med:6.0f} mm [{i_p5:6.0f}, {i_p95:6.0f}]')
        print(f'  Quad extrap:   {q_med[0]:6.0f} mm [{q_lo:6.0f}, {q_hi:6.0f}]')
        print(f'  Hybrid:        {h_med:6.0f} mm [{h_p5:6.0f}, {h_p95:6.0f}]')

    # Component breakdown at 2100
    idx_2100 = np.argmin(np.abs(hr['years'] - 2100))
    print(f'\n  Component breakdown at 2100 (medians):')
    print(f'    Thermosteric:     {hr["thermo_med"][idx_2100]:6.0f} mm')
    print(f'    Glaciers:         {hr["glacier_med"][idx_2100]:6.0f} mm')
    print(f'    Greenland:        {hr["greenland_med"][idx_2100]:6.0f} mm')
    print(f'    West Antarctica:  {hr["wais_med"][idx_2100]:6.0f} mm')
    print(f'    Peninsula:        {hr["peninsula_med"][idx_2100]:6.0f} mm')
    print(f'    TWS:              {hr["tws_med"][idx_2100]:6.0f} mm')

print('\n' + '=' * 80)